In [2]:
import pandas as pd
import numpy as np
import sys
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score
from pathlib import Path
import importlib

In [3]:
# prepare path
base = Path.cwd().parent
data_path = base / 'dataset' / 'train.csv'

In [48]:
# import custom transformer(s) and function(s)
from utils.helper import ColumnSelector, reset_helper, display_scores, ColumnDropper

In [6]:
score_data = pd.read_csv(data_path)

In [7]:
score_data_cp = score_data.copy()
X = score_data_cp.drop(columns=[
    'id',
    'exam_score'
])
y = score_data_cp['exam_score'].copy()

## Get Started
- Start with a 'dumb' model that predict every score the median score
- The result is not very good because the prediction is typically off by alomst 19 points on the scale from 0 to 100. This is almost 19% of the full-scale
- The next step is to use K-fold cross validation to see if the RMSE is consistently bad.

In [62]:
# baseline model to compare the linear regresison model againstl
# the baseline model predicts the score of every student as the mean score
baseline = DummyRegressor(strategy='mean')
baseline.fit(X, y)
baseline_predictions = baseline.predict(X)
baseline_rmse = root_mean_squared_error(baseline_predictions, y)
print(f'Baseline RMSE: {baseline_rmse}')

Baseline RMSE: 18.916869132918187


## Obversvation
- The RMSE values across the different folds are consistently close to 19. This suggests that the baseline model performs consistently poorly across different subsets of the training data.
- The next step is to try a slightly more complex model: Linear Regression.
- Before moving to Linear Regression, we build a preprocessing pipeline that scales the numerical features and encodes the categorical features.
- Scaling puts the numerical features on comparable scales and makes the preprocessing pipeline more suitable for models that are sensitive to feature scale. Categorical features need to be encoded because Linear Regression requires numerical input.

In [68]:
baseline_scores = cross_val_score(baseline, X, y, cv=5, scoring="neg_root_mean_squared_error")
display_scores(-baseline_scores)

Scores:  [18.92937996 18.93029964 18.87288703 18.87308245 18.9785329 ]
Mean:  18.916836397973306
Stadard deviation:  0.03997752508313183


In [ ]:
# build a number pipeline that: selects only numerical columns and
# do standard scaling on the columns 
numerical_pipeline = Pipeline(
    [
        ('selector', ColumnSelector(type_list=['number'])),
        ('std_scaler', StandardScaler())
    ]
)

In [ ]:
# build a categorical pipeline that: selects only categorical columns
# and do one hot encoding on the columns
categorical_pipeline = Pipeline(
    [
        ('selector', ColumnSelector(type_list=['str'])),
        ('1hot_encoder', OneHotEncoder(sparse_output=False))
    ]
)

In [51]:
# join the pipeline
combined_pipeline = FeatureUnion(
    transformer_list=[
            ('num_pipeline', numerical_pipeline),
            ('cat_pipeline', categorical_pipeline)
    ]
)

In [52]:
# create a pipeline for the model
# so that the preprocessing information does not get
# leaked between folds during cross validation
lin_pipeline = Pipeline(
    [
        ("preprocessor", combined_pipeline),
        ("linear_regression", LinearRegression())
    ]
)

In [53]:
# update: use the model pipeline to transform
# and fit to the data
lin_pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('linear_regression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformer_list transformer_list: list of (str, transformer) tuplesList of transformer objects to be applied to the data. The firsthalf of each tuple is the name of the transformer. The transformer canbe 'drop' for it to be ignored or can be 'passthrough' for features tobe passed unchanged... versionadded:: 1.1 Added the option `""passthrough""`... versionchanged:: 0.22 Deprecated `None` as a transformer in favor of 'drop'.","[('num_pipeline', ...), ('cat_pipeline', ...)]"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer.Keys are transformer names, values the weights.Raises ValueError if key not present in ``transformer_list``.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, default=TrueIf True, :meth:`get_feature_names_out` will prefix all feature nameswith the name of the transformer that generated that feature.If False, :meth:`get_feature_names_out` will not prefix any featurenames and will error if feature names are not unique... versionadded:: 1.5",True
,columns_to_drop,[]
,type_list,['number']


In [70]:
# evaluate the model
score_predictions = lin_pipeline.predict(X)
lin_rmse = root_mean_squared_error(score_predictions, y)
print(lin_rmse)

8.894405137041227


In [67]:
# cross validation
lin_scores = cross_val_score(
    lin_pipeline,
    X,
    y,
    scoring='neg_root_mean_squared_error',
    cv=5
)
display_scores(lin_scores)

Scores:  [-8.88494696 -8.90894064 -8.88063254 -8.90448784 -8.89484747]
Mean:  -8.894771088900459
Stadard deviation:  0.010877586293155432


## Quick glance

| Configuration | Mean RMSE | Std. Dev.
|---|---:|---:|
|Baseline | 18.92 | 0.04 |
|Linear Regression | 4.08 | 0.15 |

- The scores for linear regression model is much better than the baseline model.
- We can try some other ways to see if the score can improve.

In [60]:
display_scores(-scores)

Scores:  [8.88494696 8.90894064 8.88063254 8.90448784 8.89484747]
Mean:  8.894771088900459
Stadard deviation:  0.010877586293155432


In [28]:
# load test data (dataset is pre-split into the training set and test set)
score_data_test = pd.read_csv(base / 'dataset' / 'test.csv')
score_data_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 270000 entries, 0 to 269999
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                270000 non-null  int64  
 1   age               270000 non-null  int64  
 2   gender            270000 non-null  str    
 3   course            270000 non-null  str    
 4   study_hours       270000 non-null  float64
 5   class_attendance  270000 non-null  float64
 6   internet_access   270000 non-null  str    
 7   sleep_hours       270000 non-null  float64
 8   sleep_quality     270000 non-null  str    
 9   study_method      270000 non-null  str    
 10  facility_rating   270000 non-null  str    
 11  exam_difficulty   270000 non-null  str    
dtypes: float64(3), int64(2), str(7)
memory usage: 24.7 MB


In [29]:
score_data_test_cp = score_data_test.copy()

In [ ]:
# preprocessed and predict the test data
X_test = score_data_test_cp.drop(columns=['id'])
X_test_id = score_data_test_cp['id'].copy().to_numpy()
X_test_predictions = lin_pipeline.predict(X_test)

In [31]:
# write the result into a csv file
id_exam_score_combined = np.column_stack([X_test_id, X_test_predictions])
results = pd.DataFrame(id_exam_score_combined, columns=['id', 'exam_score'])
results['id'] = results['id'].astype('int32')
result_path = base / 'results'
result_path.mkdir(exist_ok=True)
results.to_csv(base / 'results' / 'results.csv', index=False)

In [32]:

results.info()

<class 'pandas.DataFrame'>
RangeIndex: 270000 entries, 0 to 269999
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          270000 non-null  int32  
 1   exam_score  270000 non-null  float64
dtypes: float64(1), int32(1)
memory usage: 3.1 MB
